# G Range


## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [G Range](#g-range)
  - [Backtesting](#g-range-backtesting)
  - [Grid search](#g-range-grid-search)
  - [Walk-forward analysis](#g-range-walk-forward)
  - [Monte Carlo simulations](#g-range-monte-carlo)
  - [Live signals](#g-range-live-signals)


Model: range / channel fade

When two horizontal levels bracket price, buy near the lower edge and sell near the upper edge of a wide-enough channel, targeting the far edge with a cushion.

Default level detector: cluster_level (pivot_level is also recommended)

How the G Range algorithm determines entry/exit:
- Detects horizontal S/R via the engine.levels detectors, selectable via the level_detector knob; at each bar the active levels are split into the nearest edge below and above the close.
- The channel must be at least g_range_min_width_atr of ATR wide to be worth fading.
- Long entry from the bottom zone of the channel (g_range_entry_zone) when the low tests the lower edge.
- Short entry from the top zone when the high tests the upper edge.
- Stop: structural, beyond the near edge (g_range_stop_buffer_atr); target: the far edge minus a cushion of the channel width (g_range_target_cushion). Exit preset structural.
- Look-ahead free: a level is usable only from its confirmation bar and until its causal invalidation bar.

## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search, oracle_ceiling
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("g_range")   # engine/strategy_configurator.py (GRangeParams — this strategy's class)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> the strategy's assigned default (exit_policy_for)

### Manual

*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
# e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_OVERRIDES = {}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic GRangeParams().
# A foreign key raises TypeError here, not a silent no-op.
# e.g. {"g_range_min_width_atr": 6.0}  or  {"level_detector": "cluster_level"}
STRATEGY_OVERRIDES = {}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = the strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["structural_rr3"]()
# Custom: from engine.exits import CompositeExit, StructuralStop, RrTarget
#         EXIT_POLICY = CompositeExit(StructuralStop(), RrTarget(3.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
# e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADE_OVERRIDES = {}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [ ]:
# Prepare the final inputs the rest of the notebook uses. Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

_window = (f"{DATA_CONFIG.start} → {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
tc = TRADING_CONFIG
_exits = (", ".join(f"{k!r}: {v!r}" for k, v in STRATEGY_CONFIG.EXITS.items())
          if EXIT_POLICY is None else f"override → {EXIT_POLICY}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} → {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Trade: initial_equity={tc.initial_equity}, position_size_bps={tc.position_size_bps}, "
      f"leverage={tc.leverage}, sizing_mode={tc.sizing_mode.value!r}, "
      f"risk_per_trade_bps={tc.risk_per_trade_bps}, direction={tc.direction.value!r}")
print("Strategy Parameters: "
      + ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(STRATEGY_CONFIG).items()))
print(f"Strategy exits: {_exits}")

<a id="g-range"></a>
## G Range

<a id="g-range-backtesting"></a>
### Backtesting

In [ ]:
# Import G Range strategy
from engine.strategies import GRangeStrategy
STRATEGY = GRangeStrategy

In [ ]:
# Backtest G Range strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="g_range")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# G Range strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="g-range-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: leave a grid out to hold those knobs fixed. Keep grids tight.
# The strategy grid sweeps level_detector so all three detectors are compared head-to-head.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"level_detector": ["pivot_level", "cluster_level", "touch_level"], "g_range_min_width_atr": [3.0, 4.0, 6.0]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "structural_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob: flip between Sharpe / P&L / profit_factor /
# any grid_search column without editing the plot. Best value per level_detector × g_range_min_width_atr
# cell, across any other swept dimension. Renders only when the strategy grid is swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"g_range_min_width_atr", "level_detector"}.issubset(gs.columns):
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="level_detector", columns="g_range_min_width_atr", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="g_range_min_width_atr", y="level_detector", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs level_detector × g_range_min_width_atr swept in the grid above — nothing to plot.")

<a id="g-range-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is the in-sample window swept for the best params; TEST_BARS is the
# out-of-sample window the winner is tested on. OBJECTIVE is any sweep metric.

GRID = {"level_detector": ["pivot_level", "cluster_level", "touch_level"], "g_range_min_width_atr": [3.0, 4.0, 6.0]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary a lot across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tends to overfit, unlikely to work next window.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
  .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits) over the strategy's base-config chart.
# Each fold re-optimises on its train window; see wf.folds_frame() for the per-fold params.
build_chart(strategy.prepare(df), trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | {SYMBOL} {INTERVAL}m").show()

<a id="g-range-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just what happened.
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="g-range-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome), runs via the notebook cell, not the CLI
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop, prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal:
  python -m engine --strategy g_range --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped


In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel / use the Stop button